# AutoMed v2 — Sistema Multiagente de Consulta Médica Experta
## Powered by AutoGen (AG2) 0.14.1 + Ollama Local + Python 3.14

**Versión:** 2.1 (Corregida para AutoGen 0.14.1 — AG2)  
**Framework:** AutoGen (AG2) 0.14.1 — Multi-agent collaborative system  
**LLM:** Ollama local (qwen2.5:7b principal, llama3.1:8b opcional)  
**Lenguaje:** 100% Español  
**Python:** 3.14 (env-llm-314) ✅  

---

### ¿Qué es AutoMed v2?

Un sistema de IA **multiagente** que simula una consulta médica experta mediante la **colaboración inteligente** de 5 agentes especializados:

1. **Medical Info Agent** → Recupera conocimiento médico
2. **Symptom Analyzer** → Analiza síntomas y diagnósticos diferenciales
3. **Treatment Suggester** → Propone tratamientos personalizados
4. **Safety Validator** → Verifica seguridad y detecta hallucinations
5. **User Interface Agent** → Mantiene conversación natural

**Cambios en v2.1:**
- ✅ AutoGen (AG2) 0.14.1 (corregido desde 0.10.0+)
- ✅ `system_message` en lugar de `system_prompt` (parámetro correcto)
- ✅ Cliente Ollama nativo: `api_type: "ollama"` (no simular OpenAI)
- ✅ `human_input_mode="NEVER"` para agentes 100% automáticos
- ✅ Uso de `ChatResult.summary` para capturar respuestas reales
- ✅ Benchmarking con conteo real de turnos de conversación

⚠️ **DISCLAIMER:** Este sistema NO es sustituto de consulta médica real.

## Celda 1: Verificación de Entorno Python 3.14

Verificamos que estamos en Python 3.14 y que las dependencias están disponibles.

In [1]:
import sys
import subprocess

print(f"[INFO] Python version: {sys.version}")
print(f"[INFO] Python executable: {sys.executable}")

# Verificar Python 3.14
if sys.version_info >= (3, 14):
    print("[OK] Python 3.14 ✓")
else:
    print(f"[WARN] Python {sys.version_info.major}.{sys.version_info.minor} detectado")
    print("[WARN] Se recomienda Python 3.14 (env-llm-314)")
'''
# Verificar dependencias clave
deps_requeridas = {
    "autogen": "AutoGen (multiagente)",
    "pandas": "Pandas (análisis datos)",
    "matplotlib": "Matplotlib (gráficos)",
    "requests": "Requests (HTTP)",
    "configparser": "ConfigParser (config.ini)"
}

print("\n[INFO] Verificando dependencias:")
missing = []
for pkg, desc in deps_requeridas.items():
    try:
        __import__(pkg)
        if pkg == "autogen":
            import autogen
            print(f"  ✓ {pkg:15} v{autogen.__version__:10} - {desc}")
        else:
            print(f"  ✓ {pkg:15}              - {desc}")
    except ImportError:
        print(f"  ✗ {pkg:15}              - {desc}")
        missing.append(pkg)

if missing:
    print(f"\n[ERROR] Dependencias faltantes: {', '.join(missing)}")
    print("[INFO] Instalar con: pip install -r requirements.txt")
    sys.exit(1)
else:
    print("\n[OK] Todas las dependencias disponibles ✓")

'''

[INFO] Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
[INFO] Python executable: c:\workspace-vc\env-llm-314\Scripts\python.exe
[OK] Python 3.14 ✓


'\n# Verificar dependencias clave\ndeps_requeridas = {\n    "autogen": "AutoGen (multiagente)",\n    "pandas": "Pandas (análisis datos)",\n    "matplotlib": "Matplotlib (gráficos)",\n    "requests": "Requests (HTTP)",\n    "configparser": "ConfigParser (config.ini)"\n}\n\nprint("\n[INFO] Verificando dependencias:")\nmissing = []\nfor pkg, desc in deps_requeridas.items():\n    try:\n        __import__(pkg)\n        if pkg == "autogen":\n            import autogen\n            print(f"  ✓ {pkg:15} v{autogen.__version__:10} - {desc}")\n        else:\n            print(f"  ✓ {pkg:15}              - {desc}")\n    except ImportError:\n        print(f"  ✗ {pkg:15}              - {desc}")\n        missing.append(pkg)\n\nif missing:\n    print(f"\n[ERROR] Dependencias faltantes: {\', \'.join(missing)}")\n    print("[INFO] Instalar con: pip install -r requirements.txt")\n    sys.exit(1)\nelse:\n    print("\n[OK] Todas las dependencias disponibles ✓")\n\n'

## Celda 2: Cargar Configuración desde config.ini

Centralizamos TODA la configuración en config.ini (UTF-8).

In [2]:
import configparser
import os
from pathlib import Path

# Cargar config.ini
config_path = "config.ini"
config = configparser.ConfigParser()

if not os.path.exists(config_path):
    print(f"[ERROR] config.ini no encontrado en {os.getcwd()}")
    print(f"[INFO] Asegúrate de ejecutar desde la carpeta del proyecto")
    sys.exit(1)

# ← CRÍTICO: UTF-8 para caracteres españoles
config.read(config_path, encoding='utf-8')
print(f"[OK] config.ini cargado correctamente (UTF-8)")

# Extraer configuración Ollama
ollama_base_url = config.get('ollama', 'base_url', fallback='http://localhost:11434')
ollama_model = config.get('ollama', 'model_name', fallback='qwen2.5:7b')
ollama_temperature = config.getfloat('ollama', 'temperature', fallback=0.2)
ollama_max_tokens = config.getint('ollama', 'max_tokens', fallback=2000)
ollama_num_ctx = config.getint('ollama', 'num_ctx', fallback=8192)

# Extraer configuración AutoGen
max_consecutive_auto_reply = config.getint('autogen', 'max_consecutive_auto_reply', fallback=10)
autogen_max_turns = config.getint('autogen', 'max_turns', fallback=8)

# Extraer configuración benchmark
benchmark_enabled = config.getboolean('benchmark', 'enabled', fallback=True)
variant_label = config.get('benchmark', 'variant_label', fallback='').strip()
if not variant_label:
    variant_label = f"{ollama_model}_t{ollama_temperature}_ctx{ollama_num_ctx}"

# Extraer configuración seguridad médica
require_disclaimer = config.getboolean('seguridad_medica', 'disclaimer', fallback=True)
#max_confidence_medical = config.getfloat('seguridad_medica', 'max_confidence_medical', fallback=0.85)
max_confidence_medical = 0.85
print(f"\n[CONFIG LOADED]")
print(f"  Ollama URL: {ollama_base_url}")
print(f"  Modelo: {ollama_model}")
print(f"  Temperatura: {ollama_temperature}")
print(f"  Max tokens: {ollama_max_tokens}")
print(f"  Context window: {ollama_num_ctx}")
print(f"  Benchmark enabled: {benchmark_enabled}")
print(f"  Variant label: {variant_label}")
print(f"  Require disclaimer: {require_disclaimer}")

[OK] config.ini cargado correctamente (UTF-8)

[CONFIG LOADED]
  Ollama URL: http://localhost:11434
  Modelo: qwen2.5:7b
  Temperatura: 0.3
  Max tokens: 2000
  Context window: 8192
  Benchmark enabled: True
  Variant label: # vacío = autogenera como {modelo}_t{temperatura}_agents{n}
  Require disclaimer: True


## Celda 3: Verificar Conectividad con Ollama

ANTES de crear agentes, verificamos que Ollama esté corriendo.

In [3]:
import requests

try:
    resp = requests.get(f"{ollama_base_url}/api/tags", timeout=5)
    if resp.status_code == 200:
        models = resp.json().get('models', [])
        print(f"[OK] Ollama conectado. Modelos disponibles:")
        '''for m in models:
            model_name = m.get('name')
            model_size = m.get('size', 0) / 1e9  # Convertir a GB
            print(f"     - {model_name} ({model_size:.1f}GB)")'''
    else:
        print(f"[ERROR] Ollama respondio con status {resp.status_code}")
        sys.exit(1)
except requests.exceptions.ConnectionError:
    print(f"[ERROR] No se pudo conectar a Ollama en {ollama_base_url}")
    print(f"[INFO] Ejecuta en otra terminal: ollama serve")
    sys.exit(1)

# Verificar modelo específico
model_names = [m.get('name') for m in models]
if ollama_model in model_names:
    print(f"[OK] Modelo '{ollama_model}' disponible ✓")
else:
    print(f"[WARN] Modelo '{ollama_model}' NO encontrado")
    print(f"[INFO] Modelos disponibles: {', '.join(model_names)}")
    print(f"[INFO] Descargar con: ollama pull {ollama_model}")
    sys.exit(1)

print(f"\n[OK] Ollama listo para usar ✓")

[OK] Ollama conectado. Modelos disponibles:
[OK] Modelo 'qwen2.5:7b' disponible ✓

[OK] Ollama listo para usar ✓


## Celda 4: Definir Agentes Especializados (AutoGen 0.14.1 — AG2)

**AutoGen (AG2) 0.14.1 tiene:**
- `ConversableAgent` para agentes conversacionales
- `system_message` (NO `system_prompt`) para instrucciones de rol
- Cliente Ollama nativo vía `api_type: "ollama"` (más robusto que simular OpenAI)
- `human_input_mode="NEVER"` para agentes 100% automáticos

In [4]:
def guardar_en_markdown(contenido: str, nombre_archivo: str = "reporte.md") -> str:
    """Guarda el texto proporcionado en un archivo con formato Markdown."""
    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write(contenido)
    return f"Archivo {nombre_archivo} guardado correctamente."

In [5]:
from autogen import ConversableAgent
import json
from datetime import datetime

# AG2 0.14.1 — Usar OpenAI-compatible API (Ollama soporta esto)
# Cambio: en lugar de "api_type": "ollama", usar "api_type": "openai"
# con base_url que apunte a Ollama /v1 endpoint
base_url_openai_compatible = ollama_base_url.rstrip('/') + '/v1'

config_list = [
    {
        "api_type": "openai",  # ← Cambio: OpenAI-compatible (Ollama lo soporta)
        "model": ollama_model,
        "base_url": base_url_openai_compatible,  # ← Con /v1 al final
        "api_key": "not-needed",  # ← Ollama no requiere API key
        "temperature": ollama_temperature,
    }
]

#print("[INFO] Configuración AutoGen para Ollama (OpenAI-compatible API):")
#print(f"[INFO] API Type: openai (Ollama endpoint compatible)")
#print(f"[INFO] Base URL: {base_url_openai_compatible}")
#print(f"[INFO] Modelo: {ollama_model}")
print(json.dumps(config_list[0], indent=2, ensure_ascii=False))

# ===========================================================================
# Agent 1: Interfaz de Usuario
# ===========================================================================
user_interface_agent = ConversableAgent(
    name="InterfazDeUsuario",
    system_message="""Eres un asistente médico amable y profesional.
Tu rol es:
1. Recibir síntomas del usuario de forma natural
2. Hacer preguntas de seguimiento para entender mejor
3. Coordinar con otros especialistas para análisis profundo
4. Presentar recomendaciones de forma clara y accesible
5. Genera la salida en formato markdown

""",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
    human_input_mode="NEVER",
)

# ===========================================================================
# Agent 2: Analizador de Síntomas
# ===========================================================================
symptom_analyzer = ConversableAgent(
    name="AnalizadorDeSintomas",
    system_message="""Eres un experto en Analisis y diagnóstico clínico con 20 años de experiencia.
Tu rol es:
1. Analizar el patrón de síntomas presentados
2. Generar diagnósticos diferenciales (lista de posibles diagnósticos)
3. Priorizar por probabilidad y urgencia
4. Identificar síntomas de alerta que requieren atención inmediata

Responde en formato claro:
- Diagnósticos principales (con confianza 0-100%)
- Síntomas de alerta detectados
- Necesidad de evaluación urgente: SI/NO
""",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
    human_input_mode="NEVER",
)

# ===========================================================================
# Agent 3: Especialista en Información Médica
# ===========================================================================
medical_info_agent = ConversableAgent(
    name="EspecialistaEnInformacionMedica",
    system_message="""Eres un especialista en medicina con acceso a base de conocimiento médico.
Tu rol es:
1. Proporcionar información factual sobre condiciones médicas
2. Explicar mecanismos de enfermedades
3. Describir síntomas típicos y variaciones
4. Compartir estadísticas relevantes de prevalencia

Sé preciso, cita fuentes si es posible, y evita especulación.
""",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
    human_input_mode="NEVER",
)

# ===========================================================================
# Agent 4: Sugeridor de Tratamientos
# ===========================================================================
treatment_suggester = ConversableAgent(
    name="SugeridorDeTratamientos",
    system_message="""Eres un especialista en opciones terapéuticas y tratamientos.
Tu rol es:
1. Proponer opciones de tratamiento basadas en diagnósticos
2. Comparar eficacia, riesgos y beneficios de cada opción
3. Considerar el contexto del paciente (edad, otras condiciones)
4. Clasificar por urgencia: inmediato / pronto / seguimiento

Formato de respuesta:
- Opción 1: [tratamiento] - Eficacia X%, Riesgos: [lista]
- Opción 2: ...
- Recomendacion: [opción preferida con justificacion]
""",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
    human_input_mode="NEVER",
)

# ===========================================================================
# Agent 5: Validador de Seguridad Médica
# ===========================================================================
safety_validator = ConversableAgent(
    name="ValidadorSeguridadMedica",
    system_message="""Eres un especialista en seguridad médica y validación de recomendaciones.
Tu rol CRÍTICO es:
1. Detectar y marcar cualquier información médica incorrecta o alucinación
2. Verificar que las recomendaciones sean seguras
3. Identificar contraindicaciones potenciales
4. Evaluar si se necesita intervención de profesional inmediata

IMPORTANTE: Marca cualquier información dudosa con [VERIFICAR CON PROFESIONAL]
Nunca recomiendes medicamentos sin advertencia de consulting médico real.
""",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
    human_input_mode="NEVER",
)

print("\n[OK] 5 agentes especializados creados:")
print(f"  1. {user_interface_agent.name}")
print(f"  2. {symptom_analyzer.name}")
print(f"  3. {medical_info_agent.name}")
print(f"  4. {treatment_suggester.name}")
print(f"  5. {safety_validator.name}")
print(f"\n[INFO] Cada agente usa: {ollama_model} via Ollama (OpenAI-compatible API)")
print(f"[INFO] Temperatura: {ollama_temperature} (determinístico para medicina)")


{
  "api_type": "openai",
  "model": "qwen2.5:7b",
  "base_url": "http://localhost:11434/v1",
  "api_key": "not-needed",
  "temperature": 0.3
}

[OK] 5 agentes especializados creados:
  1. InterfazDeUsuario
  2. AnalizadorDeSintomas
  3. EspecialistaEnInformacionMedica
  4. SugeridorDeTratamientos
  5. ValidadorSeguridadMedica

[INFO] Cada agente usa: qwen2.5:7b via Ollama (OpenAI-compatible API)
[INFO] Temperatura: 0.3 (determinístico para medicina)


In [6]:
import requests
import json

# DEBUG: Verificar qué ve Ollama vs qué intenta AutoGen
print("[DEBUG] Diagnóstico de conectividad Ollama ← AutoGen")

# 1. Verificar modelos disponibles en Ollama

try:
    resp = requests.get(f"{ollama_base_url}/api/tags", timeout=5)
    if resp.status_code == 200:
        # Verificar si el modelo configurado existe
        if ollama_model in model_names:
            print(f"[OK] ✓ Modelo '{ollama_model}' ENCONTRADO en Ollama")
        else:
            print(f"[WARN] ✗ Modelo '{ollama_model}' NO ENCONTRADO")
            print(f"[WARN] Nombres disponibles: {model_names}")
    else:
        print(f"[ERROR] Ollama respondió con status {resp.status_code}")
except Exception as e:
    print(f"[ERROR] No se pudo conectar a Ollama: {e}")

# 2. Intentar una llamada de prueba DIRECTA a Ollama (sin AutoGen)
 

# 3. Verificar el formato de la config_list que AutoGen recibirá
print("\n[DEBUG] Config que recibirán los agentes:")
config_list_debug = [
    {
        "api_type": "ollama",
        "model": ollama_model,
        "base_url": ollama_base_url,
        "temperature": ollama_temperature,
    }
]
print(json.dumps(config_list_debug[0], indent=2, ensure_ascii=False))

[DEBUG] Diagnóstico de conectividad Ollama ← AutoGen
[OK] ✓ Modelo 'qwen2.5:7b' ENCONTRADO en Ollama

[DEBUG] Config que recibirán los agentes:
{
  "api_type": "ollama",
  "model": "qwen2.5:7b",
  "base_url": "http://localhost:11434",
  "temperature": 0.3
}


## Celda 5: Función para Ejecutar Consulta Multiagente

Orquesta la colaboración entre agentes para responder una consulta médica.

In [7]:
import time
from datetime import datetime
from benchmark import medir_y_registrar  # ← IMPORTAR desde benchmark.py

def ejecutar_consulta_medica(sintomas_descripcion):
    """
    Ejecuta una consulta médica multiagente.
    
    Args:
        sintomas_descripcion (str): Descripcion de síntomas del usuario
    
    Returns:
        dict: Resultado con respuesta y métricas de benchmarking
    """
    print(f"\n{'='*70}")
    print(f"CONSULTA MEDICA MULTIAGENTE")
    print(f"{'='*70}")
    print(f"Síntomas: {sintomas_descripcion}")
    print(f"Timestamp Entrada: {datetime.now().isoformat()}")
    print(f"{'='*70}\n")
    
    # Iniciar cronometro
    t_start = time.perf_counter()
    
    # Crear mensaje inicial para agentes
    consulta_sistema = f"""
CONSULTA MEDICA:
Paciente reporta los siguientes síntomas: {sintomas_descripcion}

Por favor, analicen colaborativamente:
1. Analizador: ¿Cuales son los diagnósticos diferenciales?
2. Especialista Médico: ¿Que información relevante existe sobre estas condiciones?
3. Sugeridor: ¿Cuales son las opciones de tratamiento?
4. Validador: ¿Hay alguna contraindicación o alucinación?
5. Interfaz: Resume la recomendacion final de forma clara y con disclaimer legal.
"""
    
    try:
        # Iniciar conversación multiagente
        # En AutoGen (AG2) 0.14.1, initiate_chat devuelve ChatResult
        chat_result = user_interface_agent.initiate_chat(
            symptom_analyzer,
            message=consulta_sistema,
            max_turns=autogen_max_turns,  # Máximo de turnos
        )
        
        wall_time = time.perf_counter() - t_start
        respuesta_final = chat_result.summary or "Sin resumen disponible."
        print(f"\n[OK] Respuesta_final :" + respuesta_final)
        print(f"\n[OK] Consulta completada en {wall_time:.2f} segundos")
        print(f"[INFO] Turnos de conversación: {len(chat_result.chat_history)}\n")
        
        # USAR medir_y_registrar() desde benchmark.py — NUNCA definir inline
        metrica = medir_y_registrar(
            chat_result, 
            wall_time, 
            ollama_model, 
            ollama_temperature, 
            variant_label, 
            sintomas_descripcion
        )
        
        return {
            "respuesta": respuesta_final,
            "metrica": metrica,
            "tiempo_segundos": wall_time
        }
        
    except Exception as e:
        wall_time = time.perf_counter() - t_start
        print(f"[ERROR] Consulta falló: {e}")
        
        # medir_y_registrar() maneja excepciones automáticamente
        metrica = {
            "timestamp": datetime.now().isoformat(),
            "variant_label": variant_label,
            "consulta": sintomas_descripcion[:100],
            "exito": False,
            "error": str(e)[:200],
            "wall_time_seconds": round(wall_time, 3)
        }
        
        return {
            "respuesta": f"Error en consulta: {e}",
            "metrica": metrica,
            "tiempo_segundos": wall_time
        }

## Celda 6: Ejecutar Consultas de Prueba

Casos de prueba con síntomas reales en español.

In [8]:
# Casos de prueba (consultas médicas en español)
casos_prueba = [    
    "Siento mi cuerpo inflamado, abdomen distendido y cosquilleo en las piernas"
]

resultados = []
metricas = []

print("[INFO] Ejecutando consultas de prueba...\n")

for i, caso in enumerate(casos_prueba, 1):
    print(f"\n[CASO {i}/{len(casos_prueba)}]")
    print(f"Síntomas: {caso}")
    
    resultado = ejecutar_consulta_medica(caso)
    resultados.append(resultado)
    metricas.append(resultado['metrica'])
    
    # Mostrar resumen
    print(f"Tiempo: {resultado['tiempo_segundos']:.2f}s")
    print(f"Status: {'✓ Éxito' if resultado['metrica']['exito'] else '✗ Error'}")
    print(f"Respuesta Consulta:" + resultado['respuesta'])

print(f"\n[OK] {len(resultados)} consultas completadas")

[INFO] Ejecutando consultas de prueba...


[CASO 1/1]
Síntomas: Siento mi cuerpo inflamado, abdomen distendido y cosquilleo en las piernas

CONSULTA MEDICA MULTIAGENTE
Síntomas: Siento mi cuerpo inflamado, abdomen distendido y cosquilleo en las piernas
Timestamp Entrada: 2026-09-20T18:29:45.393294

InterfazDeUsuario (to AnalizadorDeSintomas):


CONSULTA MEDICA:
Paciente reporta los siguientes síntomas: Siento mi cuerpo inflamado, abdomen distendido y cosquilleo en las piernas

Por favor, analicen colaborativamente:
1. Analizador: ¿Cuales son los diagnósticos diferenciales?
2. Especialista Médico: ¿Que información relevante existe sobre estas condiciones?
3. Sugeridor: ¿Cuales son las opciones de tratamiento?
4. Validador: ¿Hay alguna contraindicación o alucinación?
5. Interfaz: Resume la recomendacion final de forma clara y con disclaimer legal.


--------------------------------------------------------------------------------
[autogen.oai.client: 09-20 18:30:59] {734} WARNING - Model 

## Celda 7: Persistir Métricas de Benchmarking (JSONL)

Guardamos en JSONL append-only para análisis posterior.

In [9]:
from benchmark import guardar_metricas  # ← IMPORTAR desde benchmark.py

# Guardar métricas en JSONL (append-only, UTF-8)
output_dir = "outputs"

# USAR guardar_metricas() desde benchmark.py — NUNCA definir inline
guardar_metricas(metricas, output_dir, enabled=benchmark_enabled)

guardar_en_markdown(resultado['respuesta'],f"{output_dir}/diagnostico_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md" )

# Mostrar resumen
exitosas = [m for m in metricas if m.get('exito', False)]
if exitosas:
    tiempos = [m['wall_time_seconds'] for m in exitosas]
    print(f"\n[RESUMEN BENCHMARKING]")
    print(f"  Consultas exitosas: {len(exitosas)}/{len(metricas)}")
    print(f"  Tiempo promedio: {sum(tiempos)/len(tiempos):.2f}s")
    print(f"  Tiempo mínimo: {min(tiempos):.2f}s")
    print(f"  Tiempo máximo: {max(tiempos):.2f}s")
else:
    print("[INFO] Benchmarking deshabilitado o sin consultas exitosas")

[OK] 1 métrica(s) anexada(s) a: outputs\benchmark.jsonl

[RESUMEN BENCHMARKING]
  Consultas exitosas: 1/1
  Tiempo promedio: 961.49s
  Tiempo mínimo: 961.49s
  Tiempo máximo: 961.49s


## Celda 9: Generar Reporte de Resultados

Consolidamos resultados en un reporte legible.

In [10]:
from datetime import datetime

# Generar reporte
reporte_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
reporte_filename = os.path.join(output_dir, f"reporte_{reporte_timestamp}.txt")

with open(reporte_filename, 'w', encoding='utf-8') as f:
    f.write("="*70 + "\n")
    f.write("REPORTE DE CONSULTAS MEDICAS MULTIAGENTE\n")
    f.write("AutoMed v2 - Sistema de IA Colaborativa (Python 3.14)\n")
    f.write("="*70 + "\n\n")
    
    f.write(f"Fecha: {datetime.now().isoformat()}\n")
    f.write(f"Modelo: {ollama_model}\n")
    f.write(f"Temperatura: {ollama_temperature}\n")
    f.write(f"Variante: {variant_label}\n")
    f.write(f"AutoGen version: 0.10.0+\n")
    f.write(f"Python: 3.14\n")
    f.write(f"\n")
    
    for i, resultado in enumerate(resultados, 1):
        f.write(f"\n{'='*70}\n")
        f.write(f"CONSULTA {i}\n")
        f.write(f"{'='*70}\n")
        f.write(f"Resultado Consultas: {resultado['respuesta']}s\n")
        f.write(f"Tiempo: {resultado['tiempo_segundos']:.2f}s\n")
        f.write(f"Status: {'✓ Éxito' if resultado['metrica']['exito'] else '✗ Error'}\n")
        if not resultado['metrica']['exito']:
            f.write(f"Error: {resultado['metrica'].get('error', 'Desconocido')}\n")
        f.write(f"\n")

print(f"[OK] Reporte guardado en: {reporte_filename}")
print(f"\n[INFO] Archivos generados:")
print(f"  - {reporte_filename}")
print(f"  - outputs/benchmark.jsonl")
print(f"  - outputs/latency_analysis.png")

[OK] Reporte guardado en: outputs\reporte_20260920_184546.txt

[INFO] Archivos generados:
  - outputs\reporte_20260920_184546.txt
  - outputs/benchmark.jsonl
  - outputs/latency_analysis.png


## Celda 11: Referencias y Documentación

Recursos para aprender más.

In [11]:
'''print("\n" + "="*70)
print("REFERENCIAS Y DOCUMENTACION")
print("="*70 + "\n")

print("📚 DOCUMENTOS DEL PROYECTO:")
print("  - README.md: Visión general del proyecto")
print("  - REPORTE_TECNOLOGIAS.md: Explicación detallada de tecnologías")
print("  - config.ini: Toda la configuración parametrizable")
print("  - checkpoint/CHECKPOINTS.md: Historial de hitos")

print("\n🔗 ENLACES EXTERNOS:")
print("  - AutoGen (AG2) Docs: https://docs.ag2.ai/ (proyecto AG2 — el que usamos)")
print("  - Ollama: https://ollama.ai/")
print("  - Python 3.14: https://www.python.org/")

print("\n⚙️ CONFIGURACION ACTUAL (PYTHON 3.14):")
print(f"  - Framework: AutoGen (AG2) 0.14.1")
print(f"  - Cliente Ollama: nativo (api_type: 'ollama')")
print(f"  - LLM: {ollama_model} (vía Ollama local)")
print(f"  - Temperatura: {ollama_temperature} (determinístico para medicina)")
print(f"  - Contexto: {ollama_num_ctx} tokens")
print(f"  - Max tokens (num_predict): {ollama_max_tokens}")
print(f"  - Python: 3.14 (env-llm-314)")

print("\n💡 MEJORAS EN V2.1 (Correcciones):")
print("  ✓ AutoGen (AG2) 0.14.1 — versión correcta (verificada en vivo)")
print("  ✓ `system_message` — parámetro correcto (NO `system_prompt`)")
print("  ✓ `api_type: 'ollama'` — cliente nativo (NO `'open_ai'`)")
print("  ✓ `human_input_mode='NEVER'` — agentes 100% automáticos")
print("  ✓ `ChatResult.summary` — captura respuestas reales")
print("  ✓ Conteo real de turnos — benchmarking preciso")

print("\n🎓 NOTA SOBRE VERSIONES:")
print("  - El paquete instalado es 'autogen' (AG2 — https://ag2.ai/)")
print("  - NO es 'pyautogen' (paquete distinto en PyPI)")
print("  - requirements.txt debe especificar 'autogen>=0.14.0'")

print("\n" + "="*70)'''

'print("\n" + "="*70)\nprint("REFERENCIAS Y DOCUMENTACION")\nprint("="*70 + "\n")\n\nprint("📚 DOCUMENTOS DEL PROYECTO:")\nprint("  - README.md: Visión general del proyecto")\nprint("  - REPORTE_TECNOLOGIAS.md: Explicación detallada de tecnologías")\nprint("  - config.ini: Toda la configuración parametrizable")\nprint("  - checkpoint/CHECKPOINTS.md: Historial de hitos")\n\nprint("\n🔗 ENLACES EXTERNOS:")\nprint("  - AutoGen (AG2) Docs: https://docs.ag2.ai/ (proyecto AG2 — el que usamos)")\nprint("  - Ollama: https://ollama.ai/")\nprint("  - Python 3.14: https://www.python.org/")\n\nprint("\n⚙️ CONFIGURACION ACTUAL (PYTHON 3.14):")\nprint(f"  - Framework: AutoGen (AG2) 0.14.1")\nprint(f"  - Cliente Ollama: nativo (api_type: \'ollama\')")\nprint(f"  - LLM: {ollama_model} (vía Ollama local)")\nprint(f"  - Temperatura: {ollama_temperature} (determinístico para medicina)")\nprint(f"  - Contexto: {ollama_num_ctx} tokens")\nprint(f"  - Max tokens (num_predict): {ollama_max_tokens}")\nprint(f"  

#Chatbot de salud mental utilizando la biblioteca AutoGen.
<table border="1" cellspacing="0" align="left">
    <thead>
        <tr>
            <th>Agent</th>
            <th>Role</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td><strong>AgentePaciente</strong></td>
            <td>Captures user input (mood, stress level, emotional concerns).</td>
        </tr>
        <tr>
            <td><strong>AgenteAnalisisEmocional</strong></td>
            <td>Identifies emotions based on user input.</td>
        </tr>
        <tr>
            <td><strong>AgenteRecomendacionTerapia</strong></td>
            <td>Provides relaxation techniques and coping strategies.</td>
        </tr>
    </tbody>
</table>

In [12]:
import autogen
from autogen import ConversableAgent, GroupChatManager

# Create AI Agents with distinct roles 
patient_agent = ConversableAgent(
    name="Paciente",
    system_message="Describe sus emociones y sus preocupaciones sobre la salud mental.",
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
)

emotion_analysis_agent = ConversableAgent(
    name="AnalisisEmociones",
    system_message='''Analizas las emociones del usuario basándote en lo que expresa.
-No ofrezcas tratamientos ni consejos de autocuidado.
-En su lugar, limítate a resumir las emociones predominantes que pueda estar experimentando.''',
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
)

therapy_recommendation_agent = ConversableAgent(
    name="RecomendacionTerapia",
    system_message='''Sugiere técnicas de relajación y métodos de autocuidado basándote únicamente en el análisis del Agente de Análisis de Emociones.
No analices las emociones; limítate a ofrecer recomendaciones basadas en la respuesta anterior.''',
    llm_config={"config_list": config_list},
    max_consecutive_auto_reply=max_consecutive_auto_reply,
)

# Create GroupChat for AI Agents 
groupchat = autogen.GroupChat(
    agents=[emotion_analysis_agent, therapy_recommendation_agent],
    messages=[], 
    max_round=3,  # Ensures the conversation does not stop too early 
    speaker_selection_method="round_robin"
)

# Create GroupChatManager 
manager = autogen.GroupChatManager(name="manager", groupchat=groupchat)

# Function to start the chatbot interaction 
def start_mental_health_chat():
    """Runs a chatbot for mental health support with distinct agent roles.""" 
    print("\nBienvenido al Chatbot AI Salud Mental!") 
    user_feelings = input("Como te sientes hoy ?")

    # Initiate conversation
    print("\nAnalizando Emociones...")
    response = patient_agent.initiate_chat(
        manager, 
        message=f"Me he estado sitiendo {user_feelings}. Puede ayudarme?"
    )

    # Ensure the therapy agent gets triggered
    if not response:  # If the initial response is empty, retry with explicit therapy agent prompt
        response = therapy_recommendation_agent.initiate_chat(
            manager, 
            message="Por favor, proporcione recomendaciones terapéuticas basadas en las emociones del usuario."
        )

# Run the chatbot 
#start_mental_health_chat()